# Reproduce the original PoliGraph model training

Run this notebook in Google Colab with a GPU runtime. It preserves the paper's training process: the NER corpus combines 50,000 generated sentences with the 200-policy `s_dev` distillation cohort, and purpose classification uses the manually annotated 200-phrase SetFit corpus. Training data and outputs live in Google Drive, not Git.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import shutil
ROOT = Path('/content/drive/MyDrive/poligraph-training')
SOURCE = ROOT / 'source'
NER = ROOT / 'ner'
PURPOSE = ROOT / 'purpose'
OUTPUT = ROOT / 'outputs'
for path in (SOURCE, NER, PURPOSE, OUTPUT):
    path.mkdir(parents=True, exist_ok=True)
POLICHECK_ARCHIVE = SOURCE / 'policheck-dataset-202303.tar.xz'
print('Published source archive:', POLICHECK_ARCHIVE)
print('Unpublished NER gold set:', NER / 'test.spacy')
print('Manually annotated purpose splits:', PURPOSE / 'train.jsonl', PURPOSE / 'test.jsonl')

In [ ]:
import subprocess
import sys
import torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > GPU before training.'
print(torch.cuda.get_device_name(0))
REPO = Path('/content/PoliGraph')
if (REPO / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
elif REPO.exists():
    raise RuntimeError(f'{REPO} exists but is not a Git checkout; rename or remove it before retrying.')
else:
    subprocess.run(['git', 'clone', 'https://github.com/lukeblevins/PoliGraph.git', str(REPO)], check=True)
%cd /content/PoliGraph
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'models/colab-requirements.txt', '-e', '.'], check=True)
subprocess.run([sys.executable, '-m', 'spacy', 'download', 'en_core_web_trf'], check=True)
with (OUTPUT / 'environment.txt').open('w') as environment:
    subprocess.run([sys.executable, '-m', 'pip', 'freeze'], check=True, stdout=environment)

## Build the paper-documented NER training corpus

Upload `policheck-dataset-202303.tar.xz` from the authorized UCI download to `MyDrive/poligraph-training/source/`. This cell expands the repository's original templates, obtains the actor vocabulary using the original Wikidata script, and runs the original generator over all 200 `s_dev` policies. It does not replace the training process with a compatibility dataset.

In [ ]:
if not POLICHECK_ARCHIVE.is_file():
    raise FileNotFoundError(
        f'Upload the authorized UCI archive to {POLICHECK_ARCHIVE}. '
        'The archive is not committed to Git because access requires accepting the dataset terms.'
    )
EXTRACTED = Path('/content/policheck-dataset-202303')
if not (EXTRACTED / 's_dev').is_dir():
    EXTRACTED.mkdir(parents=True, exist_ok=True)
    subprocess.run(['tar', '-xJf', str(POLICHECK_ARCHIVE), '-C', str(EXTRACTED)], check=True)
s_dev = sorted(path for path in (EXTRACTED / 's_dev').iterdir() if path.is_dir())
if len(s_dev) != 200:
    raise RuntimeError(f'Expected the paper-documented 200 s_dev policies; found {len(s_dev)}')
NER_SOURCE = Path('/content/ner-training-source')
NER_SOURCE.mkdir(parents=True, exist_ok=True)
subprocess.run([sys.executable, 'models/named-entity-recognition/expand_templates.py', 'models/named-entity-recognition/template.yml', str(NER_SOURCE / 'templates.list')], check=True)
subprocess.run([sys.executable, 'models/named-entity-recognition/expand_templates.py', 'models/named-entity-recognition/data_types.yml', str(NER_SOURCE / 'data_types.list')], check=True)
subprocess.run([sys.executable, 'models/named-entity-recognition/get_actor_entity_list.py', str(NER_SOURCE / 'actor_entities.list')], check=True)
rule_rehearsal = NER_SOURCE / 'rule_rehearsal'
if rule_rehearsal.exists() or rule_rehearsal.is_symlink():
    rule_rehearsal.unlink() if rule_rehearsal.is_symlink() else shutil.rmtree(rule_rehearsal)
rule_rehearsal.symlink_to(EXTRACTED / 's_dev', target_is_directory=True)
subprocess.run([sys.executable, 'models/named-entity-recognition/gen_ner_data.py', '--synthetic', '50000', str(NER_SOURCE)], check=True)
shutil.copy2(NER_SOURCE / 'dataset.spacy', NER / 'train.spacy')
print('Generated:', NER / 'train.spacy')

## Train and evaluate the privacy-policy NER model

The paper evaluated against 200 manually annotated text segments (about 620 sentences). That gold corpus is not included in the public archives. Export the verified annotations as `MyDrive/poligraph-training/ner/test.spacy`; do not derive it from `s_dev`, the pretrained model, or the generated training corpus.

In [ ]:
NER_TRAIN = NER / 'train.spacy'
NER_TEST = NER / 'test.spacy'
if not NER_TEST.is_file():
    raise FileNotFoundError(
        f'Missing the paper-described, manually annotated NER evaluation corpus: {NER_TEST}. '
        'It is not present in the UCI public archives and must not be synthesized.'
    )
subprocess.run([sys.executable, '-m', 'spacy', 'init', 'fill-config', 'models/named-entity-recognition/base_config.cfg', '/content/ner-config.cfg'], check=True)
subprocess.run([sys.executable, '-m', 'spacy', 'debug', 'data', '/content/ner-config.cfg', '--paths.train', str(NER_TRAIN), '--paths.dev', str(NER_TEST)], check=True)
subprocess.run([sys.executable, '-m', 'spacy', 'train', '/content/ner-config.cfg', '--gpu-id', '0', '--output', str(OUTPUT / 'ner-run'), '--paths.train', str(NER_TRAIN), '--paths.dev', str(NER_TEST)], check=True)
subprocess.run([sys.executable, '-m', 'spacy', 'evaluate', str(OUTPUT / 'ner-run/model-best'), str(NER_TEST), '--gpu-id', '0', '--output', str(OUTPUT / 'ner-metrics.json')], check=True)

## Train and evaluate purpose classification

The paper manually annotated 200 purpose phrases with the five multi-label categories and trained SetFit. The public archives contain the finished model but not those annotations. Place the reviewed JSONL splits in Drive; the notebook intentionally will not turn zero-shot or pretrained-model output into ground truth.

In [ ]:
missing_purpose = [path for path in (PURPOSE / 'train.jsonl', PURPOSE / 'test.jsonl') if not path.is_file()]
if missing_purpose:
    raise FileNotFoundError(
        'Missing manually reviewed purpose annotations: '
        + ', '.join(map(str, missing_purpose))
        + ". The UCI archives do not include the paper's 200 annotated phrases; "
          'do not substitute model-generated labels.'
    )
subprocess.run([sys.executable, 'models/purpose-classification/train_setfit.py', str(PURPOSE / 'train.jsonl'), str(PURPOSE / 'test.jsonl'), str(OUTPUT / 'purpose-model'), '--metrics-output', str(OUTPUT / 'purpose-metrics.json')], check=True)

## Promotion gate

Review `ner-metrics.json`, `purpose-metrics.json`, and `environment.txt`. The paper reports an NER baseline of 96.1% precision and 89.4% recall and purpose-classification macro precision/recall of 91.0%/94.8%. Promote only after held-out results meet the agreed baseline. After approval, package the two model directories as a versioned GitHub Release asset and update `fetch_data.py`; do not commit model binaries.